# Baseline

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb

import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel("Risk data.xlsx")

#check the distribution of labels
print("\nLabel distribution:")
print(df['label'].value_counts())
print("\nLabel distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
#prepare the data
X = df['text'].values
y = df['label'].values

#split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=4213, stratify=y
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Positive cases in train: {sum(y_train)} ({sum(y_train)/len(y_train)*100:.1f}%)")
print(f"Positive cases in test: {sum(y_test)} ({sum(y_test)/len(y_test)*100:.1f}%)")

In [ ]:
#vectorize the text data using tfidf
print("\nVectorizing text data...")
vectorizer = TfidfVectorizer(
    max_features=5000,  #use top 5000 features
    ngram_range=(1, 2),  #use unigrams and bigrams
    min_df=2,  #ignore terms that appear in less than 2 documents
    max_df=0.95,  #ignore terms that appear in more than 95% of documents
    stop_words='english',
    lowercase=True,
    strip_accents='unicode'
)

#fit and transform the training data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

In [ ]:
#train random forest model
print("\n" + "="*50)
print("Training Random Forest Model")
print("="*50)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',
    random_state=4213,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)

#make predictions and get probabilities
y_pred_rf = rf_model.predict(X_test_tfidf)
y_prob_rf = rf_model.predict_proba(X_test_tfidf)[:, 1]

#evaluate random forest
print("\nRandom Forest Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)

In [ ]:
# XGBoost model
print("\n" + "="*50)
print("Training XGBoost Model")
print("="*50)

xgb_model = xgb.XGBClassifier(
    max_depth=6,                # Control the complexity of the model
    learning_rate=0.1,          # Step size shrinkage
    n_estimators=100,           # Number of trees
    scale_pos_weight=1,         # Handle class imbalance
    random_state=42,
    eval_metric="logloss",      # Evaluation metric for binary classification
    use_label_encoder=False     # Avoid label encoder warnings
)

xgb_model.fit(X_train_tfidf, y_train)

# Make predictions and get probabilities
y_pred_xgb = xgb_model.predict(X_test_tfidf)
y_prob_xgb = xgb_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluate
print("\nXGBoost Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)


In [ ]:
#train logistic regression model
print("\n" + "="*50)
print("Training Logistic Regression Model")
print("="*50)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  #handle class imbalance
    random_state=42,
    solver='liblinear'
)

lr_model.fit(X_train_tfidf, y_train)

#make predictions and get probabilities
y_pred_lr = lr_model.predict(X_test_tfidf)
y_prob_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

#evaluate logistic regression
print("\nLogistic Regression Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

# MentalBERT

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
"""
#load dataset
ds = load_dataset("jingjietan/sdcnl-suicide")
df = pd.DataFrame(ds['train'])
"""

In [ ]:
#data preprocessing
df = df[df['text'].str.strip() != '']
print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
#prepare data
X = df['text'].values
y = df['label'].values

#split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
from huggingface_hub import login
login(new_session=False)

#enter your token

In [ ]:
from transformers import pipeline

pipe = pipeline("fill-mask", model="mental/mental-bert-base-uncased")

In [ ]:
#initialize mental bert model
print("\nLoading Mental BERT model...")
model_name = 'mental/mental-bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()  #set to evaluation mode

In [ ]:
#function to get embeddings
def get_embeddings(texts, batch_size=16):
    """
    Extract Mental Bert embeddings for texts
    Uses mean pooling of last hidden states
    """
    embeddings = []

    #process in batches for memory efficiency
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch_texts = texts[i:i+batch_size]

        #tokenize
        encoded = tokenizer(
            batch_texts.tolist() if isinstance(batch_texts, np.ndarray) else batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        #get embeddings
        with torch.no_grad():
            outputs = model(**encoded)
            #use mean pooling of last hidden states
            attention_mask = encoded['attention_mask']
            hidden_states = outputs.last_hidden_state

            #apply attention mask for mean pooling
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask

            embeddings.append(mean_embeddings.cpu().numpy())

    return np.vstack(embeddings)

#extract embeddings for training data
print("\nExtracting MentalBERT embeddings for training data...")
X_train_mentalbert = get_embeddings(X_train, batch_size=16)
print(f"Training embeddings shape: {X_train_mentalbert.shape}")

#extract embeddings for test data
print("\nExtracting MentalBERT embeddings for test data...")
X_test_mentalbert = get_embeddings(X_test, batch_size=16)
print(f"Test embeddings shape: {X_test_mentalbert.shape}")

In [ ]:
#train random forest with embeddings
print("\n" + "="*50)
print("Training Random Forest with MentalBERT Embeddings")
print("="*50)

rf_embeddings = RandomForestClassifier(
    n_estimators=200,  #more trees for complex features
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=4213,
    n_jobs=-1
)

rf_embeddings.fit(X_train_mentalbert, y_train)

#predictions
y_pred_rf_embeddings = rf_embeddings.predict(X_test_mentalbert)
y_prob_rf_embeddings = rf_embeddings.predict_proba(X_test_mentalbert)[:, 1]

#evaluate
print("\nRandom Forest + MentalBERT Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf_embeddings):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf_embeddings):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_embeddings, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_rf_embeddings = confusion_matrix(y_test, y_pred_rf_embeddings)
print(cm_rf_embeddings)

In [ ]:
# Train XGBoost model with MentalBERT embeddings
print("\n" + "="*50)
print("Training XGBoost with MentalBERT Embeddings")
print("="*50)

xgb_mentalbert = xgb.XGBClassifier(
    max_depth=6,                # Control the complexity of the model
    learning_rate=0.1,          # Step size shrinkage
    n_estimators=100,           # Number of trees
    scale_pos_weight=1,         # Handle class imbalance
    random_state=42,
    eval_metric="logloss",      # Evaluation metric for binary classification
    use_label_encoder=False     # Avoid label encoder warnings
)

# Fit the model on MentalBERT embeddings
xgb_mentalbert.fit(X_train_mentalbert, y_train)

# Make predictions and get probabilities
y_pred_xgb_mentalbert = xgb_mentalbert.predict(X_test_mentalbert)
y_prob_xgb_mentalbert = xgb_mentalbert.predict_proba(X_test_mentalbert)[:, 1]

# Evaluate XGBoost model with MentalBERT embeddings
print("\nXGBoost + MentalBERT Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb_mentalbert):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb_mentalbert):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_mentalbert, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_xgb_mentalbert = confusion_matrix(y_test, y_pred_xgb_mentalbert)
print(cm_xgb_mentalbert)


In [ ]:
#train logistic regression with MentalBERT embeddings for comparison
print("\n" + "="*50)
print("Training Logistic Regression with MentalBERT Embeddings")
print("="*50)

lr_embeddings = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=4213
)

lr_embeddings.fit(X_train_mentalbert, y_train)

#predictions
y_pred_lr_embeddings = lr_embeddings.predict(X_test_mentalbert)
y_prob_lr_embeddings = lr_embeddings.predict_proba(X_test_mentalbert)[:, 1]

#evaluate
print("\nLogistic Regression + MentalBERT Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr_embeddings):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr_embeddings):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr_embeddings, target_names=['Not at Risk', 'At Risk']))

# TD IDF

In [ ]:
#ensemble with tf-idf features
print("\n" + "="*50)
print("Hybrid Approach: MentalBERT + TF-IDF Features")
print("="*50)

from sklearn.feature_extraction.text import TfidfVectorizer

#create tf-idf features
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#combine embeddings and tf-idf features
X_train_combined = np.hstack([X_train_mentalbert, X_train_tfidf.toarray()])
X_test_combined = np.hstack([X_test_mentalbert, X_test_tfidf.toarray()])

print(f"Combined features shape: {X_train_combined.shape}")

In [ ]:
#train random forest on combined features
rf_combined = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=4213,
    n_jobs=-1
)

rf_combined.fit(X_train_combined, y_train)

#predictions
y_pred_combined = rf_combined.predict(X_test_combined)
y_prob_combined = rf_combined.predict_proba(X_test_combined)[:, 1]

#evaluate
print("\nRandom Forest + MentalBERT + TF-IDF Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_combined):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_combined):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_combined, target_names=['Not at Risk', 'At Risk']))

In [ ]:
# Train XGBoost model on combined features
print("\n" + "="*50)
print("Training XGBoost with Combined Features (MentalBERT + TF-IDF)")
print("="*50)

xgb_combined = xgb.XGBClassifier(
    max_depth=6,                # Control the complexity of the model
    learning_rate=0.1,          # Step size shrinkage
    n_estimators=100,           # Number of trees
    scale_pos_weight=1,         # Handle class imbalance
    random_state=42,
    eval_metric="logloss",      # Evaluation metric for binary classification
    use_label_encoder=False     # Avoid label encoder warnings
)

# Fit the model on combined features
xgb_combined.fit(X_train_combined, y_train)

# Make predictions and get probabilities
y_pred_combined_xgb = xgb_combined.predict(X_test_combined)
y_prob_combined_xgb = xgb_combined.predict_proba(X_test_combined)[:, 1]

# Evaluate XGBoost model with combined features
print("\nXGBoost + MentalBERT + TF-IDF Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_combined_xgb):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_combined_xgb):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_combined_xgb, target_names=['Not at Risk', 'At Risk']))
print("\nConfusion Matrix:")
cm_combined_xgb = confusion_matrix(y_test, y_pred_combined_xgb)
print(cm_combined_xgb)


In [ ]:
lr_combined = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=4213
)

lr_combined.fit(X_train_combined, y_train)

#predictions
y_pred_combined = lr_combined.predict(X_test_combined)
y_prob_combined = lr_combined.predict_proba(X_test_combined)[:, 1]

#evaluate
print("\nLogistic Regression + MentalBERT + TF-IDF Results:")
print("-"*30)
print(f"Accuracy: {accuracy_score(y_test, y_pred_combined):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_combined):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_combined, target_names=['Not at Risk', 'At Risk']))